In [ ]:
import os
from glob import glob
import numpy as np
import torch
from tqdm.auto import tqdm # Vẫn giữ tqdm.auto để thanh load đẹp trên Notebook

# ==============================================================================
# CẤU HÌNH ĐƯỜNG DẪN
# ==============================================================================
HS_ROOT = "./layer_analysis/6_method_tsd_2/hidden_states"
MAX_SAMPLES = 200 # Đặt thành None nếu muốn chạy toàn bộ

FOLDER_TO_LABEL = {
    "teacher":      "Teacher",
    "student_base": "Student-base",
    "tsd":          "TSD-KD",
    "amid":         "AMID",
    "csd":          "CSD",
    "nnm_ours":     "NNM (ours)",
}

# ==============================================================================
# TÍNH TOÁN CORE
# ==============================================================================
@torch.no_grad()
def calculate_batched_frob_nuc_norm(hs: torch.Tensor) -> list[float]:
    device = "cuda" if torch.cuda.is_available() else "cpu"
    L, seq_len, D = hs.shape

    if seq_len > 512:
        g = torch.Generator()
        g.manual_seed(seq_len * D)
        idx = torch.randperm(seq_len, generator=g)[:512]
        hs_sub = hs[:, idx, :] 
    else:
        hs_sub = hs

    hs_sub = hs_sub.to(device, dtype=torch.float32)
    hs_sub = hs_sub - hs_sub.mean(dim=1, keepdim=True)
    C = torch.matmul(hs_sub, hs_sub.mT)
    eigvals = torch.linalg.eigvalsh(C).cpu()
    
    sample_metrics = []
    for lid in range(L):
        eig = eigvals[lid]
        eig = eig[eig > 1e-10]
        if eig.numel() == 0:
            sample_metrics.append(0.0)
            continue
            
        sigma = torch.sqrt(eig)
        nuc_norm = sigma.sum().item()
        frob_norm = torch.sqrt(eig.sum()).item() 
        
        if frob_norm > 1e-12:
            sample_metrics.append(nuc_norm / frob_norm)
        else:
            sample_metrics.append(0.0)
        
    return sample_metrics

def process_model_folder(model_dir: str, max_samples: int = None) -> np.ndarray:
    pt_files = sorted(glob(os.path.join(model_dir, "sample_*.pt")))
    if not pt_files:
        return None

    if max_samples is not None:
        pt_files = pt_files[:max_samples]

    all_metrics = []
    for fpath in tqdm(pt_files, desc=f"Processing {os.path.basename(model_dir)}", leave=False):
        try:
            hs = torch.load(fpath, map_location="cpu", weights_only=True)
            sample_metrics = calculate_batched_frob_nuc_norm(hs)
            all_metrics.append(sample_metrics)
        except Exception:
            continue

    if not all_metrics:
        return None
    return np.mean(all_metrics, axis=0)

# ==============================================================================
# CHẠY VÀ IN KẾT QUẢ TEXT
# ==============================================================================
def run_analysis():
    if not os.path.isdir(HS_ROOT):
        print(f"Không tìm thấy thư mục: {HS_ROOT}")
        return

    subdirs = sorted([d for d in os.listdir(HS_ROOT) if os.path.isdir(os.path.join(HS_ROOT, d))])
    print(f"[*] Tìm thấy {len(subdirs)} model folders.\n")

    results = {}
    for folder in subdirs:
        label = FOLDER_TO_LABEL.get(folder.lower(), folder)
        model_dir = os.path.join(HS_ROOT, folder)
        
        avg_metrics = process_model_folder(model_dir, MAX_SAMPLES)
        if avg_metrics is not None:
            results[label] = avg_metrics
        else:
            print(f"⚠ Không có data hợp lệ trong: {label}")

    if not results:
        print("Không có kết quả để hiển thị.")
        return

    # Khúc này thay vì tạo bảng Pandas thì chỉ dùng text format
    model_names = list(results.keys())
    n_layers = len(list(results.values())[0])

    header = f"{'Layer':<8} |"
    for name in model_names:
        header += f" {name:<15} |"
        
    print("\n" + "=" * len(header))
    print(header)
    print("-" * len(header))
    
    for i in range(n_layers):
        layer_name = f"L{i}" if i > 0 else "Emb"
        row_str = f"{layer_name:<8} |"
        for name in model_names:
            val = results[name][i]
            row_str += f" {val:<15.4f} |"
        print(row_str)
        
    print("=" * len(header) + "\n")

# Gọi hàm
run_analysis()

[*] Tìm thấy 3 model folders.



Processing amid:   0%|          | 0/100 [00:00<?, ?it/s]

Processing nnm_ours:   0%|          | 0/100 [00:00<?, ?it/s]

Processing student-base:   0%|          | 0/100 [00:00<?, ?it/s]


Layer    | AMID            | NNM (ours)      | student-base    |
----------------------------------------------------------------
Emb      | 9.1510          | 9.1510          | 9.1510          |
L1       | 10.9507         | 10.8463         | 10.9137         |
L2       | 1.9860          | 4.8728          | 1.7817          |
L3       | 1.5147          | 1.8452          | 1.4540          |
L4       | 1.5880          | 1.9486          | 1.5164          |
L5       | 1.6506          | 2.0413          | 1.5721          |
L6       | 1.6088          | 1.9643          | 1.5347          |
L7       | 1.6364          | 2.0074          | 1.5607          |
L8       | 1.6772          | 2.0710          | 1.5957          |
L9       | 1.7006          | 2.1117          | 1.6164          |
L10      | 1.7319          | 2.1681          | 1.6468          |
L11      | 1.7329          | 2.1690          | 1.6477          |
L12      | 1.7500          | 2.1971          | 1.6609          |
L13      | 1.7745       

: 